# Paper-style OmniXAS Ti Training

Train/load the paper model types for Ti:

- **UniversalXAS**: one FEFF model trained on Ti–Cu FEFF data
- **ExpertXAS**: target-only Ti FEFF / Ti VASP models
- **Tuned-UniversalXAS**: UniversalXAS fine-tuned on Ti FEFF / Ti VASP

Paper: Kharel *et al.*, arXiv:2409.19552.

## Paper settings used

- M3GNet transfer features: **64** inputs
- XAS spectra: **141** outputs
- XASBlock: Linear → BatchNorm → SiLU → Dropout; Softplus output
- Adam + MSE
- Max epochs: **1000**
- Early stopping: **50 epochs** without validation improvement (`check_val_every_n_epoch=2`, `patience=25`)
- Tuned-UniversalXAS starts from UniversalXAS and fine-tunes on target data
- Tuned dropout sweep: **0.5** and **0.0**

## 1. Imports and settings

In [35]:
from datetime import datetime
from pathlib import Path
import os
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightning.pytorch import seed_everything

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.metrics import ModelMetrics
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

# Set to 42 for reproducible training; set to None for a fresh random seed.
SEED = 42
if SEED is None:
    SEED = random.SystemRandom().randint(0, 2**32 - 1)
RUN_SEED_LABEL = f"seed{SEED}"
os.environ["PYTHONHASHSEED"] = str(SEED)
seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision("high")
print("Using seed:", SEED)

REPO_ROOT = Path.cwd()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").exists()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not find OmniXAS repo root.")
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
OUTPUT_ROOT = REPO_ROOT / "output" / "training"
RESULTS_DIR = OUTPUT_ROOT / "comparisons" / "paper_reproduction_ti" / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{RUN_SEED_LABEL}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_DIR.exists(), f"Missing ML data: {DATA_DIR}"

INPUT_DIM, OUTPUT_DIM = 64, 141
UNIVERSAL_DIMS = [500, 500, 550]
TI_FEFF_EXPERT_DIMS = [600, 600, 450]
TI_VASP_EXPERT_DIMS = [500, 600, 400]

UNIVERSAL_BATCH = 32
TI_FEFF_BATCH = 32
TI_VASP_BATCH = 64

MAX_EPOCHS = 1000
PATIENCE = 25
INITIAL_LR = 1e-2
MIN_LR = 1e-4
DEFAULT_DROPOUT = 0.5
TUNED_DROPOUT_SWEEP = [0.5, 0.0]

# Number of independent random-seed runs per training block when a TRAIN_* flag is True.
N_TRAINING_RUNS = 1
assert N_TRAINING_RUNS >= 1

# Turn on only what you want to train.
TRAIN_UNIVERSAL_FEFF = False
TRAIN_TI_FEFF_EXPERT = False
TRAIN_TI_VASP_EXPERT = False
TRAIN_TI_FEFF_TUNED = False
TRAIN_TI_VASP_TUNED = False

CKPT_UNIVERSAL_FEFF = OUTPUT_ROOT / "universalXAS" / "All_FEFF" / "runs"
CKPT_TI_FEFF_EXPERT = OUTPUT_ROOT / "expertXAS" / "Ti_FEFF" / "runs"
CKPT_TI_VASP_EXPERT = OUTPUT_ROOT / "expertXAS" / "Ti_VASP" / "runs"
CKPT_TI_FEFF_TUNED = OUTPUT_ROOT / "tunedUniversalXAS" / "Ti_FEFF" / "runs"
CKPT_TI_VASP_TUNED = OUTPUT_ROOT / "tunedUniversalXAS" / "Ti_VASP" / "runs"

print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Seed set to 42


Using seed: 42
Repo root: /mnt/c/Users/anton/desktop/omnixas
Data dir: /mnt/c/Users/anton/desktop/omnixas/tutorial_omnixas/ml_data
Results dir: /mnt/c/Users/anton/desktop/omnixas/output/training/comparisons/paper_reproduction_ti/20260610_103723_seed42


## 2. Load ML-ready data

In [36]:
# FEFF data for UniversalXAS: all eight elements.
FEFF_ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
feff_splits = {}

for element in FEFF_ELEMENTS:
    split_data = {}
    for split_name in ["train", "val", "test"]:
        X = np.loadtxt(DATA_DIR / f"{element}_FEFF_{split_name}_X.txt", dtype=np.float32)
        y = np.loadtxt(DATA_DIR / f"{element}_FEFF_{split_name}_y.txt", dtype=np.float32)
        assert X.shape[1] == INPUT_DIM and y.shape[1] == OUTPUT_DIM
        split_data[split_name] = MLData(X=X, y=y)
    feff_splits[element] = MLSplits(**split_data)

ti_feff = feff_splits["Ti"]

# Ti VASP data for ExpertXAS/Tuned-UniversalXAS cross-fidelity case.
ti_vasp_data = {}
for split_name in ["train", "val", "test"]:
    X = np.loadtxt(DATA_DIR / f"Ti_VASP_{split_name}_X.txt", dtype=np.float32)
    y = np.loadtxt(DATA_DIR / f"Ti_VASP_{split_name}_y.txt", dtype=np.float32)
    assert X.shape[1] == INPUT_DIM and y.shape[1] == OUTPUT_DIM
    ti_vasp_data[split_name] = MLData(X=X, y=y)
ti_vasp = MLSplits(**ti_vasp_data)

# Concatenate element-wise FEFF splits for UniversalXAS.
universal_feff = MLSplits(
    train=MLData(
        X=np.concatenate([feff_splits[e].train.X for e in FEFF_ELEMENTS], axis=0),
        y=np.concatenate([feff_splits[e].train.y for e in FEFF_ELEMENTS], axis=0),
    ),
    val=MLData(
        X=np.concatenate([feff_splits[e].val.X for e in FEFF_ELEMENTS], axis=0),
        y=np.concatenate([feff_splits[e].val.y for e in FEFF_ELEMENTS], axis=0),
    ),
    test=MLData(
        X=np.concatenate([feff_splits[e].test.X for e in FEFF_ELEMENTS], axis=0),
        y=np.concatenate([feff_splits[e].test.y for e in FEFF_ELEMENTS], axis=0),
    ),
)

print("Universal FEFF train/val/test:", universal_feff.train.X.shape, universal_feff.val.X.shape, universal_feff.test.X.shape)
print("Ti FEFF train/val/test:", ti_feff.train.X.shape, ti_feff.val.X.shape, ti_feff.test.X.shape)
print("Ti VASP train/val/test:", ti_vasp.train.X.shape, ti_vasp.val.X.shape, ti_vasp.test.X.shape)

Universal FEFF train/val/test: (54967, 64) (6857, 64) (6857, 64)
Ti FEFF train/val/test: (5140, 64) (641, 64) (641, 64)
Ti VASP train/val/test: (3030, 64) (377, 64) (377, 64)


## 3. UniversalXAS: train/load all-FEFF model

In [37]:
XASBlock.DROPOUT = DEFAULT_DROPOUT

if TRAIN_UNIVERSAL_FEFF:
    universal_jobs = []
    universal_new_ckpts = []
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        train_seed_label = f"seed{train_seed}"
        run_dir = CKPT_UNIVERSAL_FEFF / f"paper_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{train_seed_label}"
        run_dir.mkdir(parents=True, exist_ok=False)
        universal_jobs.append((run_number, train_seed, run_dir, True))
else:
    universal_ckpts = sorted(CKPT_UNIVERSAL_FEFF.glob("paper_*/best*.ckpt"))
    assert universal_ckpts, f"No paper UniversalXAS checkpoints found in {CKPT_UNIVERSAL_FEFF}"
    UNIVERSAL_FEFF_CKPT = min(
        universal_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    UNIVERSAL_FEFF_DIR = UNIVERSAL_FEFF_CKPT.parent
    print("Loading UniversalXAS:", UNIVERSAL_FEFF_CKPT)
    universal_jobs = [(1, SEED, UNIVERSAL_FEFF_DIR, False)]

for run_number, train_seed, run_dir, do_train in universal_jobs:
    seed_everything(train_seed, workers=True)
    universal_feff_model = XASBlockRegressor(
        directory=str(run_dir),
        overwrite_save_dir=False,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=UNIVERSAL_DIMS,
        batch_size=UNIVERSAL_BATCH,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=PATIENCE,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
    )
    if do_train:
        print(f"Training UniversalXAS run {run_number}/{N_TRAINING_RUNS}, seed={train_seed}:", run_dir)
        universal_feff_model.fit(universal_feff)
        universal_new_ckpts.extend(sorted(run_dir.glob("best*.ckpt")))

if TRAIN_UNIVERSAL_FEFF:
    assert universal_new_ckpts, "UniversalXAS training finished but no best*.ckpt was found."
    UNIVERSAL_FEFF_CKPT = min(
        universal_new_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    UNIVERSAL_FEFF_DIR = UNIVERSAL_FEFF_CKPT.parent
    universal_feff_model.cfg.directory = str(UNIVERSAL_FEFF_DIR)
    print("Best new UniversalXAS:", UNIVERSAL_FEFF_CKPT)

universal_feff_model.load("best")

Seed set to 42
2026-06-10 10:37:35.100 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/universalXAS/All_FEFF/runs/paper_20260609_113606_310699_seed3179728593/best-model-epoch=941-val_loss=0.0077.ckpt


Loading UniversalXAS: /mnt/c/Users/anton/desktop/omnixas/output/training/universalXAS/All_FEFF/runs/paper_20260609_113606_310699_seed3179728593/best-model-epoch=941-val_loss=0.0077.ckpt


## 4. ExpertXAS: train/load Ti FEFF

In [38]:
XASBlock.DROPOUT = DEFAULT_DROPOUT

if TRAIN_TI_FEFF_EXPERT:
    ti_feff_expert_jobs = []
    ti_feff_expert_new_ckpts = []
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        train_seed_label = f"seed{train_seed}"
        run_dir = CKPT_TI_FEFF_EXPERT / f"paper_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{train_seed_label}"
        run_dir.mkdir(parents=True, exist_ok=False)
        ti_feff_expert_jobs.append((run_number, train_seed, run_dir, True))
else:
    ti_feff_expert_ckpts = sorted(CKPT_TI_FEFF_EXPERT.glob("paper_*/best*.ckpt"))
    assert ti_feff_expert_ckpts, f"No paper Ti FEFF ExpertXAS checkpoints found in {CKPT_TI_FEFF_EXPERT}"
    TI_FEFF_EXPERT_CKPT = min(
        ti_feff_expert_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_FEFF_EXPERT_DIR = TI_FEFF_EXPERT_CKPT.parent
    print("Loading Ti FEFF ExpertXAS:", TI_FEFF_EXPERT_CKPT)
    ti_feff_expert_jobs = [(1, SEED, TI_FEFF_EXPERT_DIR, False)]

for run_number, train_seed, run_dir, do_train in ti_feff_expert_jobs:
    seed_everything(train_seed, workers=True)
    ti_feff_expert_model = XASBlockRegressor(
        directory=str(run_dir),
        overwrite_save_dir=False,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=TI_FEFF_EXPERT_DIMS,
        batch_size=TI_FEFF_BATCH,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=PATIENCE,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
    )
    if do_train:
        print(f"Training Ti FEFF ExpertXAS run {run_number}/{N_TRAINING_RUNS}, seed={train_seed}:", run_dir)
        ti_feff_expert_model.fit(ti_feff)
        ti_feff_expert_new_ckpts.extend(sorted(run_dir.glob("best*.ckpt")))

if TRAIN_TI_FEFF_EXPERT:
    assert ti_feff_expert_new_ckpts, "Ti FEFF ExpertXAS training finished but no best*.ckpt was found."
    TI_FEFF_EXPERT_CKPT = min(
        ti_feff_expert_new_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_FEFF_EXPERT_DIR = TI_FEFF_EXPERT_CKPT.parent
    ti_feff_expert_model.cfg.directory = str(TI_FEFF_EXPERT_DIR)
    print("Best new Ti FEFF ExpertXAS:", TI_FEFF_EXPERT_CKPT)

ti_feff_expert_model.load("best")

Seed set to 42
2026-06-10 10:37:35.329 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/expertXAS/Ti_FEFF/runs/paper_20260609_124418_836094_seed1438024015/best-model-epoch=467-val_loss=0.0123.ckpt


Loading Ti FEFF ExpertXAS: /mnt/c/Users/anton/desktop/omnixas/output/training/expertXAS/Ti_FEFF/runs/paper_20260609_124418_836094_seed1438024015/best-model-epoch=467-val_loss=0.0123.ckpt


## 5. ExpertXAS: train/load Ti VASP

In [39]:
XASBlock.DROPOUT = DEFAULT_DROPOUT

if TRAIN_TI_VASP_EXPERT:
    ti_vasp_expert_jobs = []
    ti_vasp_expert_new_ckpts = []
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        train_seed_label = f"seed{train_seed}"
        run_dir = CKPT_TI_VASP_EXPERT / f"paper_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{train_seed_label}"
        run_dir.mkdir(parents=True, exist_ok=False)
        ti_vasp_expert_jobs.append((run_number, train_seed, run_dir, True))
else:
    ti_vasp_expert_ckpts = sorted(CKPT_TI_VASP_EXPERT.glob("paper_*/best*.ckpt"))
    assert ti_vasp_expert_ckpts, f"No paper Ti VASP ExpertXAS checkpoints found in {CKPT_TI_VASP_EXPERT}"
    TI_VASP_EXPERT_CKPT = min(
        ti_vasp_expert_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_VASP_EXPERT_DIR = TI_VASP_EXPERT_CKPT.parent
    print("Loading Ti VASP ExpertXAS:", TI_VASP_EXPERT_CKPT)
    ti_vasp_expert_jobs = [(1, SEED, TI_VASP_EXPERT_DIR, False)]

for run_number, train_seed, run_dir, do_train in ti_vasp_expert_jobs:
    seed_everything(train_seed, workers=True)
    ti_vasp_expert_model = XASBlockRegressor(
        directory=str(run_dir),
        overwrite_save_dir=False,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=TI_VASP_EXPERT_DIMS,
        batch_size=TI_VASP_BATCH,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=PATIENCE,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
    )
    if do_train:
        print(f"Training Ti VASP ExpertXAS run {run_number}/{N_TRAINING_RUNS}, seed={train_seed}:", run_dir)
        ti_vasp_expert_model.fit(ti_vasp)
        ti_vasp_expert_new_ckpts.extend(sorted(run_dir.glob("best*.ckpt")))

if TRAIN_TI_VASP_EXPERT:
    assert ti_vasp_expert_new_ckpts, "Ti VASP ExpertXAS training finished but no best*.ckpt was found."
    TI_VASP_EXPERT_CKPT = min(
        ti_vasp_expert_new_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_VASP_EXPERT_DIR = TI_VASP_EXPERT_CKPT.parent
    ti_vasp_expert_model.cfg.directory = str(TI_VASP_EXPERT_DIR)
    print("Best new Ti VASP ExpertXAS:", TI_VASP_EXPERT_CKPT)

ti_vasp_expert_model.load("best")

Seed set to 42
2026-06-10 10:37:35.548 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/expertXAS/Ti_VASP/runs/paper_20260609_130249_757841_seed971313314/best-model-epoch=329-val_loss=0.0532.ckpt


Loading Ti VASP ExpertXAS: /mnt/c/Users/anton/desktop/omnixas/output/training/expertXAS/Ti_VASP/runs/paper_20260609_130249_757841_seed971313314/best-model-epoch=329-val_loss=0.0532.ckpt


## 6. Tuned-UniversalXAS: fine-tune/load Ti FEFF

In [40]:
if TRAIN_TI_FEFF_TUNED:
    ti_feff_tuned_jobs = []
    ti_feff_tuned_new_ckpts = []
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        for dropout in TUNED_DROPOUT_SWEEP:
            dropout_label = str(dropout).replace(".", "p")
            train_seed_label = f"seed{train_seed}"
            run_dir = CKPT_TI_FEFF_TUNED / f"paper_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{train_seed_label}_dropout{dropout_label}"
            run_dir.mkdir(parents=True, exist_ok=False)
            ti_feff_tuned_jobs.append((run_number, train_seed, dropout, UNIVERSAL_FEFF_DIR, run_dir, True))
else:
    ti_feff_tuned_ckpts = sorted(CKPT_TI_FEFF_TUNED.glob("paper_*/best*.ckpt"))
    assert ti_feff_tuned_ckpts, f"No paper Ti FEFF Tuned-UniversalXAS checkpoints found in {CKPT_TI_FEFF_TUNED}"
    TI_FEFF_TUNED_CKPT = min(
        ti_feff_tuned_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_FEFF_TUNED_DIR = TI_FEFF_TUNED_CKPT.parent
    print("Loading Ti FEFF Tuned-UniversalXAS:", TI_FEFF_TUNED_CKPT)
    ti_feff_tuned_jobs = [(1, SEED, 0.0, TI_FEFF_TUNED_DIR, TI_FEFF_TUNED_DIR, False)]

for run_number, train_seed, dropout, start_dir, run_dir, do_train in ti_feff_tuned_jobs:
    seed_everything(train_seed, workers=True)
    XASBlock.DROPOUT = dropout
    ti_feff_tuned_model = XASBlockRegressor(
        directory=str(start_dir),
        overwrite_save_dir=False,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=UNIVERSAL_DIMS,
        batch_size=TI_FEFF_BATCH,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=PATIENCE,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
    )
    ti_feff_tuned_model.load("best")
    if do_train:
        print(f"Fine-tuning Ti FEFF Tuned-UniversalXAS run {run_number}/{N_TRAINING_RUNS}, seed={train_seed}, dropout={dropout}:", run_dir)
        ti_feff_tuned_model.cfg.directory = str(run_dir)
        ti_feff_tuned_model.fit(ti_feff)
        ti_feff_tuned_new_ckpts.extend(sorted(run_dir.glob("best*.ckpt")))

if TRAIN_TI_FEFF_TUNED:
    assert ti_feff_tuned_new_ckpts, "Ti FEFF Tuned-UniversalXAS training finished but no best*.ckpt was found."
    TI_FEFF_TUNED_CKPT = min(
        ti_feff_tuned_new_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_FEFF_TUNED_DIR = TI_FEFF_TUNED_CKPT.parent
    ti_feff_tuned_model.cfg.directory = str(TI_FEFF_TUNED_DIR)
    print("Best new Ti FEFF Tuned-UniversalXAS:", TI_FEFF_TUNED_CKPT)
    ti_feff_tuned_model.load("best")

Seed set to 42
2026-06-10 10:37:35.248 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/tunedUniversalXAS/Ti_FEFF/runs/paper_20260609_125836_878953_seed928445356_dropout0p5/best-model-epoch=365-val_loss=0.0119.ckpt


Loading Ti FEFF Tuned-UniversalXAS: /mnt/c/Users/anton/desktop/omnixas/output/training/tunedUniversalXAS/Ti_FEFF/runs/paper_20260609_125836_878953_seed928445356_dropout0p5/best-model-epoch=365-val_loss=0.0119.ckpt


## 7. Tuned-UniversalXAS: fine-tune/load Ti VASP

In [41]:
if TRAIN_TI_VASP_TUNED:
    ti_vasp_tuned_jobs = []
    ti_vasp_tuned_new_ckpts = []
    for run_number in range(1, N_TRAINING_RUNS + 1):
        train_seed = random.randint(0, 2**32 - 1)
        for dropout in TUNED_DROPOUT_SWEEP:
            dropout_label = str(dropout).replace(".", "p")
            train_seed_label = f"seed{train_seed}"
            run_dir = CKPT_TI_VASP_TUNED / f"paper_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}_{train_seed_label}_dropout{dropout_label}"
            run_dir.mkdir(parents=True, exist_ok=False)
            ti_vasp_tuned_jobs.append((run_number, train_seed, dropout, UNIVERSAL_FEFF_DIR, run_dir, True))
else:
    ti_vasp_tuned_ckpts = sorted(CKPT_TI_VASP_TUNED.glob("paper_*/best*.ckpt"))
    assert ti_vasp_tuned_ckpts, f"No paper Ti VASP Tuned-UniversalXAS checkpoints found in {CKPT_TI_VASP_TUNED}"
    TI_VASP_TUNED_CKPT = min(
        ti_vasp_tuned_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_VASP_TUNED_DIR = TI_VASP_TUNED_CKPT.parent
    print("Loading Ti VASP Tuned-UniversalXAS:", TI_VASP_TUNED_CKPT)
    ti_vasp_tuned_jobs = [(1, SEED, 0.0, TI_VASP_TUNED_DIR, TI_VASP_TUNED_DIR, False)]

for run_number, train_seed, dropout, start_dir, run_dir, do_train in ti_vasp_tuned_jobs:
    seed_everything(train_seed, workers=True)
    XASBlock.DROPOUT = dropout
    ti_vasp_tuned_model = XASBlockRegressor(
        directory=str(start_dir),
        overwrite_save_dir=False,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=UNIVERSAL_DIMS,
        batch_size=TI_VASP_BATCH,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=PATIENCE,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
    )
    ti_vasp_tuned_model.load("best")
    if do_train:
        print(f"Fine-tuning Ti VASP Tuned-UniversalXAS run {run_number}/{N_TRAINING_RUNS}, seed={train_seed}, dropout={dropout}:", run_dir)
        ti_vasp_tuned_model.cfg.directory = str(run_dir)
        ti_vasp_tuned_model.fit(ti_vasp)
        ti_vasp_tuned_new_ckpts.extend(sorted(run_dir.glob("best*.ckpt")))

if TRAIN_TI_VASP_TUNED:
    assert ti_vasp_tuned_new_ckpts, "Ti VASP Tuned-UniversalXAS training finished but no best*.ckpt was found."
    TI_VASP_TUNED_CKPT = min(
        ti_vasp_tuned_new_ckpts,
        key=lambda p: float(re.search(r"val_loss[=_](\d+(?:\.\d+)?)", p.name).group(1)),
    )
    TI_VASP_TUNED_DIR = TI_VASP_TUNED_CKPT.parent
    ti_vasp_tuned_model.cfg.directory = str(TI_VASP_TUNED_DIR)
    print("Best new Ti VASP Tuned-UniversalXAS:", TI_VASP_TUNED_CKPT)
    ti_vasp_tuned_model.load("best")

Seed set to 42
2026-06-10 10:37:35.480 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/tunedUniversalXAS/Ti_VASP/runs/paper_20260608_123031_402553_seed2072745802_dropout0p0/best-model-epoch=545-val_loss=0.0463.ckpt


Loading Ti VASP Tuned-UniversalXAS: /mnt/c/Users/anton/desktop/omnixas/output/training/tunedUniversalXAS/Ti_VASP/runs/paper_20260608_123031_402553_seed2072745802_dropout0p0/best-model-epoch=545-val_loss=0.0463.ckpt


## 8. Paper-style evaluation table

For each model family, this cell scans the saved `best*.ckpt` runs and selects the checkpoint with the highest eta.

In [42]:
paper_eta = {
    ("Ti FEFF", "ExpertXAS"): 6.35,
    ("Ti FEFF", "UniversalXAS"): 4.19,
    ("Ti FEFF", "Tuned-UniversalXAS"): 7.63,
    ("Ti VASP", "ExpertXAS"): 4.75,
    ("Ti VASP", "Tuned-UniversalXAS"): 5.27,
}


def median_mse(y_true, y_pred):
    return float(np.median(np.mean((y_true - y_pred) ** 2, axis=1)))


def score_checkpoint(ckpt_path, hidden_dims, batch_size, split):
    model = XASBlockRegressor(
        directory=str(ckpt_path.parent),
        overwrite_save_dir=False,
        input_dim=INPUT_DIM,
        output_dim=OUTPUT_DIM,
        hidden_dims=hidden_dims,
        batch_size=batch_size,
        max_epochs=MAX_EPOCHS,
        early_stopping_patience=PATIENCE,
        initial_lr=INITIAL_LR,
        min_lr=MIN_LR,
    )
    model.load("best")

    pred = model.predict(split.test.X)
    target = split.test.y
    metrics = ModelMetrics(predictions=pred, targets=target)

    baseline = np.repeat(split.train.y.mean(axis=0, keepdims=True), len(target), axis=0)
    baseline_median_mse = median_mse(target, baseline)
    model_median_mse = float(metrics.median_of_mse_per_spectra)

    seed_match = re.search(r"seed(\d+)", ckpt_path.parent.name)
    return {
        "notebook_seed": SEED,
        "checkpoint_seed": int(seed_match.group(1)) if seed_match else np.nan,
        "mse": float(metrics.mse),
        "median_mse": model_median_mse,
        "baseline_median_mse": baseline_median_mse,
        "eta": baseline_median_mse / model_median_mse,
    }


model_specs = [
    ("Ti FEFF", "ExpertXAS", CKPT_TI_FEFF_EXPERT, TI_FEFF_EXPERT_DIMS, TI_FEFF_BATCH, ti_feff),
    ("Ti FEFF", "UniversalXAS", CKPT_UNIVERSAL_FEFF, UNIVERSAL_DIMS, UNIVERSAL_BATCH, ti_feff),
    ("Ti FEFF", "Tuned-UniversalXAS", CKPT_TI_FEFF_TUNED, UNIVERSAL_DIMS, TI_FEFF_BATCH, ti_feff),
    ("Ti VASP", "ExpertXAS", CKPT_TI_VASP_EXPERT, TI_VASP_EXPERT_DIMS, TI_VASP_BATCH, ti_vasp),
    ("Ti VASP", "Tuned-UniversalXAS", CKPT_TI_VASP_TUNED, UNIVERSAL_DIMS, TI_VASP_BATCH, ti_vasp),
]

rows = []
for dataset, model_name, ckpt_root, hidden_dims, batch_size, split in model_specs:
    ckpts = sorted(ckpt_root.glob("paper_*/best*.ckpt"))
    assert ckpts, f"No checkpoints found in {ckpt_root}"

    scored = [score_checkpoint(ckpt, hidden_dims, batch_size, split) for ckpt in ckpts]
    best = max(scored, key=lambda row: row["eta"])
    best.update({
        "dataset": dataset,
        "model": model_name,
        "paper_eta": paper_eta[(dataset, model_name)],
    })
    rows.append(best)
    print(f"{dataset} / {model_name}: selected eta={best['eta']:.6f}")

results = pd.DataFrame(rows)[[
    "notebook_seed",
    "checkpoint_seed",
    "dataset",
    "model",
    "mse",
    "median_mse",
    "baseline_median_mse",
    "eta",
    "paper_eta",
]]

display(results)
results.to_csv(RESULTS_DIR / "paper_style_ti_results.csv", index=False)
print("Saved:", RESULTS_DIR / "paper_style_ti_results.csv")

2026-06-10 10:37:35.702 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/expertXAS/Ti_FEFF/runs/paper_20260609_124418_836094_seed1438024015/best-model-epoch=467-val_loss=0.0123.ckpt
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument

Ti FEFF / ExpertXAS: selected eta=6.763191


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
2026-06-10 10:37:40.579 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/universalXAS/All_FEFF/runs/paper_20260609_1

Ti FEFF / UniversalXAS: selected eta=5.059749


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
2026-06-10 10:37:45.364 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/tunedUniversalXAS/Ti_FEFF/runs/paper_202606

...

Ti VASP / ExpertXAS: selected eta=4.599576


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
2026-06-10 10:38:11.147 | INFO     | omnixas.model.xasblock_regressor:load:141 - Loading model from /mnt/c/Users/anton/desktop/omnixas/output/training/tunedUniversalXAS/Ti_VASP/runs/paper_202606

Ti VASP / Tuned-UniversalXAS: selected eta=5.225176


,notebook_seed,checkpoint_seed,dataset,model,mse,median_mse,baseline_median_mse,eta,paper_eta
0,42,1438024015,Ti FEFF,ExpertXAS,0.012048,0.007188,0.048614,6.763191,6.35
1,42,4067027602,Ti FEFF,UniversalXAS,0.014276,0.009608,0.048614,5.059749,4.19
2,42,3171776035,Ti FEFF,Tuned-UniversalXAS,0.011547,0.006847,0.048614,7.099719,7.63
3,42,971313314,Ti VASP,ExpertXAS,0.057186,0.037115,0.170714,4.599576,4.75
4,42,2072745802,Ti VASP,Tuned-UniversalXAS,0.052609,0.032671,0.170714,5.225176,5.27


Saved: /mnt/c/Users/anton/desktop/omnixas/output/training/comparisons/paper_reproduction_ti/20260610_103723_seed42/paper_style_ti_results.csv
